# MedBot Phase 3: RAG Models, Baseline LSTM, and Comprehensive Evaluation

This notebook implements **Phase 3** of the MedBot project with:
- **Preprocessing**: Tokenization and embedding preparation (extending Phase 2)
- **Baseline Model**: Simple LSTM retriever for cosine-similarity search
- **RAG Models**: 
  - ChatGPT (GPT-3.5-Turbo) via Emergent LLM Key
  - Llama-2-7B-chat medical-tuned via Hugging Face
- **ChromaDB**: Vector storage for retrieval augmentation
- **Evaluation**: Retrieval F1, ROUGE-1/2/L, hallucination rate, response time
- **Visualization**: Comparative plots and results export

**Environment**: Google Colab-friendly with GPU support

## 1. Environment Setup & GPU Check

In [ ]:
# Check GPU availability
import torch
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Consider using smaller models or enabling GPU in Colab (Runtime > Change runtime type > GPU)")

## 2. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch transformers datasets chromadb openai requests sentence-transformers \
    faiss-cpu numpy pandas matplotlib seaborn scikit-learn rouge-score tqdm accelerate tiktoken

## 3. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM, pipeline
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_recall_fscore_support
from rouge_score import rouge_scorer
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import openai
import time
from tqdm.auto import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 4. Configuration & API Keys

In [ ]:
# Configuration
CONFIG = {
    'device': device,
    'max_length': 512,
    'batch_size': 16,
    'embedding_dim': 384,
    'lstm_hidden_dim': 256,
    'lstm_num_layers': 2,
    'num_epochs': 5,
    'learning_rate': 0.001,
    'top_k_retrieval': 5,
    'num_eval_questions': 100,
}

# API Keys - Set via environment variables
# For ChatGPT: Using Emergent LLM Key
EMERGENT_LLM_KEY = os.environ.get('EMERGENT_LLM_KEY', 'sk-emergent-56016CcDc780e503a4')
openai.api_key = EMERGENT_LLM_KEY

# For Hugging Face models (optional, for gated models)
HF_TOKEN = os.environ.get('HUGGINGFACE_API_KEY', None)

print("✅ Configuration loaded")
print(f"✅ Emergent LLM Key configured: {EMERGENT_LLM_KEY[:15]}...")

## 5. Data Loading & Preprocessing

Loading MedQA dataset and preparing medical text corpus for retrieval.

In [ ]:
# Load MedQA dataset from Hugging Face
print("Loading MedQA dataset...")
try:
    # Using the MedQA USMLE dataset
    dataset = load_dataset("GBaker/MedQA-USMLE-4-options", split="test")
    print(f"✅ Loaded {len(dataset)} questions from MedQA")
except Exception as e:
    print(f"⚠️  Error loading MedQA: {e}")
    print("Using alternative dataset...")
    dataset = load_dataset("medmcqa", split="validation")
    print(f"✅ Loaded {len(dataset)} questions from MedMCQA")

# Sample evaluation questions
eval_questions = dataset.shuffle(seed=42).select(range(min(CONFIG['num_eval_questions'], len(dataset))))
print(f"✅ Selected {len(eval_questions)} questions for evaluation")

In [ ]:
# Prepare medical corpus for retrieval
# Using a subset of PubMed abstracts and medical knowledge
print("Loading medical corpus...")

# Option 1: Use PubMed subset (smaller, faster)
try:
    corpus_dataset = load_dataset("pubmed", split="train[:5000]")  # 5k abstracts for demo
    medical_corpus = [
        {"text": item['MedlineCitation']['Article']['Abstract']['AbstractText'][0] if item.get('MedlineCitation', {}).get('Article', {}).get('Abstract', {}).get('AbstractText') else "",
         "id": str(i),
         "title": item.get('MedlineCitation', {}).get('Article', {}).get('ArticleTitle', "Untitled")}
        for i, item in enumerate(corpus_dataset)
    ]
    medical_corpus = [doc for doc in medical_corpus if doc['text']]  # Filter empty
except Exception as e:
    print(f"⚠️  PubMed loading failed: {e}")
    print("Creating synthetic medical corpus...")
    # Fallback: Create a basic corpus from the questions themselves
    medical_corpus = [
        {"text": item['question'] if 'question' in item else item.get('sent1', ''),
         "id": str(i),
         "title": f"Medical Question {i}"}
        for i, item in enumerate(dataset)
    ]

print(f"✅ Medical corpus prepared: {len(medical_corpus)} documents")
print(f"Sample document: {medical_corpus[0]['text'][:200]}...")

## 6. Preprocessing: Tokenization & Embedding Preparation

In [ ]:
# Initialize tokenizer and embedding model
print("Initializing embedding model...")
embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'  # Lightweight, fast
embedding_model = SentenceTransformer(embedding_model_name, device=device)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print(f"✅ Embedding model loaded: {embedding_model_name}")

# Tokenize and embed corpus
print("Embedding medical corpus...")
corpus_texts = [doc['text'] for doc in medical_corpus]
corpus_embeddings = embedding_model.encode(
    corpus_texts, 
    batch_size=32, 
    show_progress_bar=True,
    convert_to_tensor=True,
    device=device
)
corpus_embeddings = corpus_embeddings.cpu().numpy()
print(f"✅ Corpus embeddings shape: {corpus_embeddings.shape}")

## 7. ChromaDB Setup for Vector Storage

In [ ]:
# Initialize ChromaDB
print("Initializing ChromaDB...")
chroma_client = chromadb.Client(Settings(
    chroma_db_impl="duckdb+parquet",
    persist_directory="./chroma_db"
))

# Create or get collection
try:
    chroma_client.delete_collection(name="medical_corpus")
except:
    pass

collection = chroma_client.create_collection(
    name="medical_corpus",
    metadata={"description": "Medical corpus for RAG retrieval"}
)

# Add documents to ChromaDB
print("Adding documents to ChromaDB...")
for i, doc in enumerate(tqdm(medical_corpus)):
    collection.add(
        embeddings=[corpus_embeddings[i].tolist()],
        documents=[doc['text']],
        ids=[doc['id']],
        metadatas=[{"title": doc['title']}]
    )

print(f"✅ ChromaDB initialized with {collection.count()} documents")

## 8. Baseline LSTM Retriever Model

In [ ]:
# Define LSTM Retriever Model
class LSTMRetriever(nn.Module):
    """Simple LSTM-based retriever for creating query embeddings"""
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, output_dim):
        super(LSTMRetriever, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)  # *2 for bidirectional
        self.dropout = nn.Dropout(0.3)
        
    def forward(self, x, lengths=None):
        # x: [batch_size, seq_len]
        embedded = self.dropout(self.embedding(x))  # [batch_size, seq_len, embedding_dim]
        
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), 
                                                       batch_first=True, enforce_sorted=False)
            lstm_out, (hidden, cell) = self.lstm(packed)
            lstm_out, _ = nn.utils.rnn.pad_packed_sequence(lstm_out, batch_first=True)
        else:
            lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # Use mean pooling over sequence
        pooled = torch.mean(lstm_out, dim=1)  # [batch_size, hidden_dim*2]
        output = self.fc(pooled)  # [batch_size, output_dim]
        return F.normalize(output, p=2, dim=1)  # L2 normalization for cosine similarity

# Initialize model
vocab_size = len(tokenizer.vocab)
lstm_model = LSTMRetriever(
    vocab_size=vocab_size,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dim=CONFIG['lstm_hidden_dim'],
    num_layers=CONFIG['lstm_num_layers'],
    output_dim=corpus_embeddings.shape[1]  # Match corpus embedding dimension
).to(device)

print(f"✅ LSTM Retriever initialized")
print(f"   Parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

In [ ]:
# Training Dataset for LSTM
class MedicalQADataset(Dataset):
    def __init__(self, questions, tokenizer, max_length=128):
        self.questions = questions
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.questions)
    
    def __getitem__(self, idx):
        question = self.questions[idx]
        if isinstance(question, dict):
            text = question.get('question', question.get('sent1', ''))
        else:
            text = str(question)
        
        encoded = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'text': text
        }

# Create training dataset
train_questions = [item for item in dataset.shuffle(seed=42).select(range(min(1000, len(dataset))))]
train_dataset = MedicalQADataset(train_questions, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)

print(f"✅ Training dataset: {len(train_dataset)} samples")

In [ ]:
# Train LSTM Retriever
def train_lstm_retriever(model, train_loader, num_epochs, learning_rate):
    """Train LSTM retriever with contrastive learning"""
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CosineEmbeddingLoss()
    
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            lengths = attention_mask.sum(dim=1)
            
            # Forward pass
            query_embeddings = model(input_ids, lengths)
            
            # Simple contrastive loss: maximize similarity with random corpus documents
            # In practice, use proper positive/negative pairs
            batch_size = query_embeddings.size(0)
            random_indices = torch.randint(0, len(corpus_embeddings), (batch_size,))
            target_embeddings = torch.from_numpy(
                corpus_embeddings[random_indices]
            ).float().to(device)
            
            # Positive labels (all 1s for simplicity)
            labels = torch.ones(batch_size).to(device)
            
            loss = criterion(query_embeddings, target_embeddings, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
    
    return model

print("Training LSTM Retriever...")
lstm_model = train_lstm_retriever(
    lstm_model, 
    train_loader, 
    CONFIG['num_epochs'], 
    CONFIG['learning_rate']
)
print("✅ LSTM training complete")

# Save model
torch.save(lstm_model.state_dict(), 'lstm_retriever_model.pt')
print("✅ Model saved: lstm_retriever_model.pt")

## 9. RAG Model Integration

### 9.1 ChatGPT RAG (via Emergent LLM Key)

In [ ]:
def retrieve_with_chromadb(query, top_k=5):
    """Retrieve relevant documents using ChromaDB"""
    query_embedding = embedding_model.encode([query], convert_to_tensor=True, device=device)
    query_embedding = query_embedding.cpu().numpy()
    
    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )
    
    retrieved_docs = []
    for i, doc in enumerate(results['documents'][0]):
        retrieved_docs.append({
            'text': doc,
            'metadata': results['metadatas'][0][i] if results['metadatas'] else {},
            'distance': results['distances'][0][i] if 'distances' in results else None
        })
    
    return retrieved_docs

def chatgpt_rag(question, retrieved_docs):
    """Generate answer using ChatGPT with retrieved context"""
    # Format context from retrieved documents
    context = "\n\n".join([f"Document {i+1}: {doc['text']}" for i, doc in enumerate(retrieved_docs)])
    
    prompt = f"""You are a medical expert. Answer the following question based on the provided context.

Context:
{context}

Question: {question}

Provide a clear, concise answer based on the context. If the context doesn't contain enough information, indicate that."""
    
    try:
        response = openai.ChatCompletion.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful medical assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.0,
            max_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"⚠️  ChatGPT API error: {e}")
        return f"Error: {str(e)}"

# Test ChatGPT RAG
test_question = "What are the symptoms of myocardial infarction?"
test_docs = retrieve_with_chromadb(test_question, top_k=3)
test_answer = chatgpt_rag(test_question, test_docs)
print("\n✅ ChatGPT RAG Test:")
print(f"Question: {test_question}")
print(f"Answer: {test_answer}")

### 9.2 Llama-2 Medical RAG

In [ ]:
# Load Llama-2 Medical Model
print("Loading Llama-2 medical model...")
print("⚠️  This may take several minutes and requires significant GPU memory (16GB+)")

try:
    # Try loading medical-tuned Llama model
    llama_model_name = "ruslanmv/Medical-Llama2-7B"  # Medical-tuned version
    
    llama_tokenizer = AutoTokenizer.from_pretrained(llama_model_name, token=HF_TOKEN)
    llama_pipeline = pipeline(
        "text-generation",
        model=llama_model_name,
        tokenizer=llama_tokenizer,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        max_new_tokens=500,
        token=HF_TOKEN
    )
    print(f"✅ Llama-2 medical model loaded: {llama_model_name}")
    
except Exception as e:
    print(f"⚠️  Llama-2 loading error: {e}")
    print("Using fallback: smaller medical model or mock responses")
    llama_pipeline = None

def llama_rag(question, retrieved_docs):
    """Generate answer using Llama-2 with retrieved context"""
    if llama_pipeline is None:
        return "[Llama model not available - using fallback response]"
    
    context = "\n\n".join([f"Context {i+1}: {doc['text']}" for i, doc in enumerate(retrieved_docs)])
    
    prompt = f"""### Context:
{context}

### Question:
{question}

### Answer:
"""
    
    try:
        output = llama_pipeline(
            prompt,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            num_return_sequences=1,
        )
        answer = output[0]['generated_text'].split('### Answer:')[-1].strip()
        return answer
    except Exception as e:
        print(f"⚠️  Llama generation error: {e}")
        return f"Error: {str(e)}"

# Test Llama RAG
if llama_pipeline:
    test_answer_llama = llama_rag(test_question, test_docs)
    print("\n✅ Llama-2 RAG Test:")
    print(f"Question: {test_question}")
    print(f"Answer: {test_answer_llama}")

### 9.3 Baseline LSTM RAG

In [ ]:
def baseline_lstm_retrieve(question, top_k=5):
    """Retrieve using trained LSTM model"""
    lstm_model.eval()
    with torch.no_grad():
        # Tokenize question
        encoded = tokenizer(
            question,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        lengths = attention_mask.sum(dim=1)
        
        # Get query embedding
        query_embedding = lstm_model(input_ids, lengths)
        query_embedding = query_embedding.cpu().numpy()
        
        # Compute cosine similarity with corpus
        similarities = np.dot(corpus_embeddings, query_embedding.T).squeeze()
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        retrieved_docs = [
            {
                'text': medical_corpus[idx]['text'],
                'similarity': float(similarities[idx]),
                'metadata': {'title': medical_corpus[idx]['title']}
            }
            for idx in top_indices
        ]
    
    return retrieved_docs

def baseline_generate_answer(question, retrieved_docs):
    """Simple answer generation based on retrieved documents"""
    # For baseline, just return the most relevant document snippet
    if not retrieved_docs:
        return "No relevant information found."
    
    answer = f"Based on the retrieved information: {retrieved_docs[0]['text'][:300]}..."
    return answer

# Test Baseline
test_docs_lstm = baseline_lstm_retrieve(test_question, top_k=3)
test_answer_baseline = baseline_generate_answer(test_question, test_docs_lstm)
print("\n✅ Baseline LSTM Test:")
print(f"Question: {test_question}")
print(f"Answer: {test_answer_baseline}")

## 10. Evaluation Metrics

Implementing Retrieval F1, ROUGE scores, hallucination detection, and response time.

In [ ]:
class RAGEvaluator:
    """Comprehensive evaluator for RAG systems"""
    def __init__(self):
        self.rouge_scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    
    def compute_retrieval_metrics(self, retrieved_docs, ground_truth_docs):
        """Compute retrieval precision, recall, F1"""
        retrieved_ids = set([doc.get('id', doc.get('text', '')[:50]) for doc in retrieved_docs])
        ground_truth_ids = set([doc.get('id', doc.get('text', '')[:50]) for doc in ground_truth_docs])
        
        if len(ground_truth_ids) == 0:
            return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        
        tp = len(retrieved_ids & ground_truth_ids)
        precision = tp / len(retrieved_ids) if len(retrieved_ids) > 0 else 0.0
        recall = tp / len(ground_truth_ids) if len(ground_truth_ids) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return {'precision': precision, 'recall': recall, 'f1': f1}
    
    def compute_rouge_scores(self, generated_answer, reference_answer):
        """Compute ROUGE-1, ROUGE-2, ROUGE-L scores"""
        scores = self.rouge_scorer.score(reference_answer, generated_answer)
        return {
            'rouge1': scores['rouge1'].fmeasure,
            'rouge2': scores['rouge2'].fmeasure,
            'rougeL': scores['rougeL'].fmeasure
        }
    
    def detect_hallucination(self, generated_answer, retrieved_docs):
        """Simple hallucination detection: check if answer contains info from retrieved docs"""
        if not retrieved_docs or not generated_answer:
            return 1.0  # High hallucination if no docs or no answer
        
        # Check token overlap between answer and retrieved documents
        answer_tokens = set(generated_answer.lower().split())
        doc_tokens = set()
        for doc in retrieved_docs:
            doc_tokens.update(doc.get('text', '').lower().split())
        
        if len(answer_tokens) == 0:
            return 1.0
        
        overlap = len(answer_tokens & doc_tokens) / len(answer_tokens)
        hallucination_rate = 1.0 - overlap  # Higher overlap = lower hallucination
        
        return hallucination_rate
    
    def measure_response_time(self, func, *args, **kwargs):
        """Measure function execution time"""
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        return result, end_time - start_time

evaluator = RAGEvaluator()
print("✅ Evaluator initialized")

## 11. Comprehensive Evaluation

Evaluating all three models (Baseline LSTM, ChatGPT RAG, Llama-2 RAG) on 100+ MedQA questions.

In [ ]:
def evaluate_rag_system(questions, system_name, retrieve_func, generate_func, evaluator):
    """Evaluate a RAG system on given questions"""
    results = []
    
    print(f"\nEvaluating {system_name}...")
    for item in tqdm(questions, desc=system_name):
        # Extract question and reference answer
        if isinstance(item, dict):
            question = item.get('question', item.get('sent1', ''))
            reference = item.get('answer', item.get('ending0', 'Unknown'))
        else:
            question = str(item)
            reference = "Unknown"
        
        if not question:
            continue
        
        try:
            # Retrieval
            retrieved_docs, retrieval_time = evaluator.measure_response_time(
                retrieve_func, question, top_k=CONFIG['top_k_retrieval']
            )
            
            # Generation
            answer, generation_time = evaluator.measure_response_time(
                generate_func, question, retrieved_docs
            )
            
            # Metrics
            rouge_scores = evaluator.compute_rouge_scores(answer, reference)
            hallucination_rate = evaluator.detect_hallucination(answer, retrieved_docs)
            total_time = retrieval_time + generation_time
            
            results.append({
                'system': system_name,
                'question': question[:100],
                'answer': answer[:200],
                'reference': reference[:200],
                'rouge1': rouge_scores['rouge1'],
                'rouge2': rouge_scores['rouge2'],
                'rougeL': rouge_scores['rougeL'],
                'hallucination_rate': hallucination_rate,
                'response_time': total_time,
                'retrieval_time': retrieval_time,
                'generation_time': generation_time
            })
        except Exception as e:
            print(f"⚠️  Error processing question: {e}")
            continue
    
    return results

# Evaluate all systems
all_results = []

# 1. Baseline LSTM
baseline_results = evaluate_rag_system(
    eval_questions,
    "Baseline LSTM",
    baseline_lstm_retrieve,
    baseline_generate_answer,
    evaluator
)
all_results.extend(baseline_results)

# 2. ChatGPT RAG
chatgpt_results = evaluate_rag_system(
    eval_questions,
    "ChatGPT RAG",
    retrieve_with_chromadb,
    chatgpt_rag,
    evaluator
)
all_results.extend(chatgpt_results)

# 3. Llama-2 RAG (if available)
if llama_pipeline:
    llama_results = evaluate_rag_system(
        eval_questions,
        "Llama-2 Medical RAG",
        retrieve_with_chromadb,
        llama_rag,
        evaluator
    )
    all_results.extend(llama_results)

print("\n✅ Evaluation complete!")
print(f"Total results: {len(all_results)}")

## 12. Results Analysis & Visualization

In [ ]:
# Convert results to DataFrame
results_df = pd.DataFrame(all_results)

# Compute aggregate statistics
aggregate_stats = results_df.groupby('system').agg({
    'rouge1': ['mean', 'std'],
    'rouge2': ['mean', 'std'],
    'rougeL': ['mean', 'std'],
    'hallucination_rate': ['mean', 'std'],
    'response_time': ['mean', 'std'],
    'retrieval_time': ['mean', 'std'],
    'generation_time': ['mean', 'std']
}).round(4)

print("\n" + "="*80)
print("AGGREGATE STATISTICS")
print("="*80)
print(aggregate_stats)

# Save results to CSV
results_df.to_csv('phase3_results.csv', index=False)
aggregate_stats.to_csv('phase3_aggregate_stats.csv')
print("\n✅ Results saved: phase3_results.csv, phase3_aggregate_stats.csv")

In [ ]:
# Visualization 1: ROUGE Scores Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['rouge1', 'rouge2', 'rougeL']
titles = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']

for ax, metric, title in zip(axes, metrics, titles):
    data = results_df.groupby('system')[metric].mean().sort_values()
    data.plot(kind='barh', ax=ax, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax.set_title(f'{title} Score Comparison', fontsize=14, fontweight='bold')
    ax.set_xlabel('Score', fontsize=12)
    ax.set_ylabel('System', fontsize=12)
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('rouge_scores_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: rouge_scores_comparison.png")

In [ ]:
# Visualization 2: Hallucination Rate
fig, ax = plt.subplots(figsize=(10, 6))
hallucination_data = results_df.groupby('system')['hallucination_rate'].mean().sort_values()
colors = ['#2ECC71' if val < 0.5 else '#E74C3C' for val in hallucination_data.values]
hallucination_data.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Hallucination Rate by System', fontsize=16, fontweight='bold')
ax.set_xlabel('Hallucination Rate (lower is better)', fontsize=12)
ax.set_ylabel('System', fontsize=12)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Threshold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('hallucination_rate_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: hallucination_rate_comparison.png")

In [ ]:
# Visualization 3: Response Time Analysis
fig, ax = plt.subplots(figsize=(12, 6))
response_time_data = results_df.groupby('system')[['retrieval_time', 'generation_time']].mean()
response_time_data.plot(kind='bar', stacked=True, ax=ax, color=['#3498DB', '#9B59B6'])
ax.set_title('Response Time Breakdown', fontsize=16, fontweight='bold')
ax.set_ylabel('Time (seconds)', fontsize=12)
ax.set_xlabel('System', fontsize=12)
ax.legend(['Retrieval Time', 'Generation Time'], loc='upper left')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('response_time_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: response_time_breakdown.png")

In [ ]:
# Visualization 4: Comprehensive Radar Chart
from math import pi

# Prepare data for radar chart
categories = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'Low Hallucination', 'Speed']
systems = results_df['system'].unique()

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

for system in systems:
    system_data = results_df[results_df['system'] == system]
    values = [
        system_data['rouge1'].mean(),
        system_data['rouge2'].mean(),
        system_data['rougeL'].mean(),
        1 - system_data['hallucination_rate'].mean(),  # Invert for "low hallucination"
        1 / (1 + system_data['response_time'].mean())  # Normalize speed
    ]
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, label=system)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=12)
ax.set_ylim(0, 1)
ax.set_title('Comprehensive RAG System Comparison', size=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.savefig('comprehensive_radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: comprehensive_radar_chart.png")

## 13. Summary & Conclusions

In [ ]:
# Generate summary report
summary = f"""
{'='*80}
MEDBOT PHASE 3 - EVALUATION SUMMARY
{'='*80}

Dataset: {len(eval_questions)} MedQA questions
Systems Evaluated: {len(results_df['system'].unique())}
  - Baseline LSTM Retriever
  - ChatGPT RAG (GPT-3.5-Turbo via Emergent LLM Key)
  - Llama-2 Medical RAG (if available)

KEY FINDINGS:
--------------
"""

for system in results_df['system'].unique():
    system_data = results_df[results_df['system'] == system]
    summary += f"\n{system}:\n"
    summary += f"  ROUGE-1: {system_data['rouge1'].mean():.4f} (±{system_data['rouge1'].std():.4f})\n"
    summary += f"  ROUGE-2: {system_data['rouge2'].mean():.4f} (±{system_data['rouge2'].std():.4f})\n"
    summary += f"  ROUGE-L: {system_data['rougeL'].mean():.4f} (±{system_data['rougeL'].std():.4f})\n"
    summary += f"  Hallucination Rate: {system_data['hallucination_rate'].mean():.4f}\n"
    summary += f"  Avg Response Time: {system_data['response_time'].mean():.3f}s\n"

summary += f"""
{'='*80}
DELIVERABLES:
{'='*80}
✅ Trained baseline LSTM model: lstm_retriever_model.pt
✅ ChromaDB vector index: ./chroma_db/
✅ Detailed results: phase3_results.csv
✅ Aggregate statistics: phase3_aggregate_stats.csv
✅ Visualizations:
   - rouge_scores_comparison.png
   - hallucination_rate_comparison.png
   - response_time_breakdown.png
   - comprehensive_radar_chart.png

{'='*80}
NEXT STEPS:
{'='*80}
1. Fine-tune LSTM on larger medical corpus
2. Implement more sophisticated hallucination detection
3. Optimize retrieval strategies (hybrid search, re-ranking)
4. Expand evaluation to larger question sets
5. Deploy best-performing model to production
"""

print(summary)

# Save summary
with open('phase3_summary.txt', 'w') as f:
    f.write(summary)
print("\n✅ Summary saved: phase3_summary.txt")

## 14. Export All Results

In [ ]:
# Create a comprehensive results package
import zipfile

files_to_zip = [
    'phase3_results.csv',
    'phase3_aggregate_stats.csv',
    'phase3_summary.txt',
    'rouge_scores_comparison.png',
    'hallucination_rate_comparison.png',
    'response_time_breakdown.png',
    'comprehensive_radar_chart.png',
    'lstm_retriever_model.pt'
]

with zipfile.ZipFile('medbot_phase3_results.zip', 'w') as zipf:
    for file in files_to_zip:
        try:
            zipf.write(file)
            print(f"✅ Added: {file}")
        except FileNotFoundError:
            print(f"⚠️  Not found: {file}")

print("\n✅ All results packaged: medbot_phase3_results.zip")
print("\n" + "="*80)
print("🎉 PHASE 3 COMPLETE!")
print("="*80)